## Customizing Agents with ClaudeAgentOptions

# Lesson 3: System Tuning and Lifecycles via `ClaudeAgentOptions`

In the previous lesson, you discovered how to implement the atomic, one-shot `query()` pattern to manage non-blocking communication pipelines. Each time you invoked `query()`, the underlying system used out-of-the-box defaults—a standard model family, an un-tailored system persona, and un-throttled turn parameters.

While these settings work for basic text exploration, building enterprise-grade tools requires tighter structural constraints. This is where `ClaudeAgentOptions` comes in—it is your configuration object for defining exactly how the runtime should behave during an execution loop.

---

## Understanding Agent Configuration with `ClaudeAgentOptions`

`ClaudeAgentOptions` is a comprehensive configuration object that controls every aspect of your agent's behavior and capabilities. While your string prompt tells the agent **what** objective to achieve, `ClaudeAgentOptions` dictates **how** the agent operates, what resources it can access, and under what constraints.

Think of it as setting the boundaries of a workspace—the prompt is the assignment, while the options define the working conditions, tools, safety permissions, and runtime limits.

### Operational Axes Managed by `ClaudeAgentOptions`:

* **Model Selection:** Choose which Claude model powers your agent.
* **Personality & System Prompts:** Inject specific roles, tone configurations, and formatting rules on top of base instructions.
* **Reasoning Throttle Limits:** Set a hard ceiling on the number of execution turns to prevent runaway looping costs.
* **Tool Registries:** Explicitly declare which local utilities (such as `Read`, `Write`, or `Bash`) are visible to the agent.
* **Permission Management:** Toggle automated tool execution or enforce strict human-in-the-loop manual approval hooks.
* **Workspace Directory Target:** Override the execution path where the local runtime subprocess operates.
* **Extensible Integrations:** Attach remote Model Context Protocol (MCP) servers to link tools like Slack, GitHub, or enterprise databases.

In this module, we will focus on three core parameters that shape the agent's baseline lifecycle: model selection, system prompts, and turn limits.

---

## Selecting Your Model Target

The `model` parameter allows you to choose which engine drives your agent's reasoning loop. Different models offer different trade-offs between speed, cost, and capability. Anthropic provides several Claude models, each optimized for different use cases.

```python
from claude_agent_sdk import ClaudeAgentOptions

# Configure the agent with a targeted model family
options = ClaudeAgentOptions(
    model="haiku"  # Selects the fast, efficient model tier
)

```

### Methods for Passing Model Identifiers:

1. **Model Family Names:** Use simple tokens like `"haiku"`, `"sonnet"`, or `"opus"`. The SDK automatically maps these to the latest stable release in that family, making them convenient for development.
2. **Version Identifiers:** Use full model identifiers (e.g., `"claude-haiku-4-5-20251001"`) to pin execution to a specific static release. This is critical for enterprise production systems where you require predictable, repeatable testing baselines.

---

## Shaping Agent Behavior with System Prompts

The `system_prompt` parameter establishes the agent's role, behavioral boundaries, and communication tone before it reviews your actual prompt.

When working with autonomous tool-using agents, the system prompt is invaluable for enforcing structural rules—such as defining linting rules for a coding assistant, setting schema rules for an analytical tool, or restricting access to certain directories.

```python
options = ClaudeAgentOptions(
    model="haiku",
    system_prompt="You are an enthusiastic, beginner-friendly programming tutor."
)

```

> ⚠️ **Under the Hood Integration:** The SDK does not replace the agent's default operational framework. Instead, it takes your custom string and blends it with the runtime's internal system prompt (which manages core tool definitions and safety controls). Your custom prompt layer sits on top, allowing you to easily adjust personality and style constraints without breaking core tool handling logic.

---

## Limiting Agent Turns with `max_turns`

The `max_turns` parameter places a hard ceiling on the number of continuous reasoning cycles the agent can execute before the session terminates.

A single turn represents one complete cycle in which the agent analyzes the context, decides to run a tool, waits for the result, evaluates the outcome, and chooses whether to proceed or stop. Setting this parameter is critical for preventing runaway token costs when an agent gets caught in an infinite error-correction loop.

```python
options = ClaudeAgentOptions(
    model="haiku",
    system_prompt="You are an enthusiastic, beginner-friendly programming tutor.",
    max_turns=5  # Strictly terminates after 5 reasoning iterations
)

```

### Handling Turn Exhaustion:

If the agent hits its `max_turns` limit before fully completing its objective:

* The session stops safely, and the generator yields a final `ResultMessage`.
* The message will contain `subtype='error_max_turns'` and `result=None`.
* Even though the task timed out, `is_error` returns `False`. The SDK treats this as a safe boundary condition rather than a system crash, and still populates the token usage and cost metrics so you can analyze why the agent ran out of turns.

---

## Passing Options to the `query()` Loop

Once you instantiate a `ClaudeAgentOptions` object, you hook it into your execution stream using the `options` keyword argument inside the `query()` function:

```python
import anyio
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock

async def main():
    # 1. Package the runtime parameters
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are an enthusiastic, beginner-friendly programming tutor.",
        max_turns=5
    )
    
    # 2. Inject the options context block into the query pipeline
    async_stream = query(
        prompt="What is the difference between Claude and Claude Code?",
        options=options
    )
    
    async for message in async_stream:
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text)

if __name__ == "__main__":
    anyio.run(main)

```

---

## Observing the Configured Agent's Response

When running this custom configuration, the agent adopts the exact tone, constraints, and structural rules you provided:

```text
Great question! Let me explain the difference between Claude and Claude Code:

## **Claude (Standard)**
Claude is Anthropic's large language model AI assistant that you interact with through various interfaces like:
- **Claude.ai** - the web chat interface
- **Claude API** - for developers integrating Claude into applications

Claude excels at general conversation, answering questions, and writing text.

## **Claude Code**
Claude Code is a **specialized development environment** built for software development. It includes capabilities like:
- **Direct file access** - Read, edit, and write files in your project
- **Terminal execution** - Run commands, scripts, and tests

| Feature | Claude | Claude Code |
| :--- | :--- | :--- |
| File manipulation | Limited | Full read/write access |
| Terminal/CLI access | None | Full bash terminal |
| Code execution | Cannot run code | Can execute code & tests |

🚀 What coding concepts should we look into next?

```

### Performance Analysis:

Because you passed the `"enthusiastic beginner-friendly tutor"` persona, the agent automatically organizes the technical response using clear tables, accessible definitions, and supportive sign-offs.

Choosing `model="haiku"` ensures the entire response stream returns rapidly and cost-effectively, while setting `max_turns=5` guarantees the engine operates safely within your preset runtime boundaries.

---

## Summary

You have mastered how to configure and tune your agent's runtime environment using `ClaudeAgentOptions`:

* The **`model`** parameter allows you to target specific model tiers to balance execution speed, intelligence, and token costs.
* The **`system_prompt`** parameter lets you mold the agent's persona, communication style, and structural constraints.
* The **`max_turns`** parameter gives you control over session lifecycles, protecting your application from runaway loops and predictable billing.

Let's head over to the practice exercises to build custom configuration structures yourself! 🚀

## Configuring Your First Agent Model

Now that you understand how ClaudeAgentOptions shapes agent behavior, it's time to configure your first agent by selecting a model.

The starter code contains a working query, but it uses the default model settings. Your task is to take control by creating a ClaudeAgentOptions object and specifying which model should power the agent.

Here's what you need to do:

    Create a ClaudeAgentOptions object and set the model parameter to "haiku" (using the simple family name).
    Pass your options object to the query() function using the options parameter.

The rest of the code is already set up to display the agent's response and show you the usage metrics, so you can see exactly how much this interaction costs. Once you complete these two steps, you'll have successfully configured your first agent with a custom model selection!

```
import anyio
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ResultMessage,
)


async def main():
    # TODO: Create a ClaudeAgentOptions object with model set to "haiku"
    
    # TODO: Pass the options object to query() using the options parameter
    async for message in query(
        prompt="What is the difference between Claude and Claude Code?"
    ):
        # Display assistant message responses
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text)
        
        # Display usage metrics
        elif isinstance(message, ResultMessage):
            print(f"\n--- Usage Metrics ---")
            print(f"Cost: ${message.total_cost_usd:.4f}")
            print(f"Input tokens: {message.usage.get('input_tokens', 0)}")
            print(f"Output tokens: {message.usage.get('output_tokens', 0)}")


if __name__ == "__main__":
    anyio.run(main)

```

Here is the completed code with `ClaudeAgentOptions` instantiated and mapped to your streaming task.

By setting `model="haiku"`, you explicitly switch the underlying processing engine to Anthropic's fastest and most cost-effective model family tier:

```python
import anyio
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ResultMessage,
)


async def main():
    # Create a ClaudeAgentOptions object with model set to "haiku"
    options = ClaudeAgentOptions(model="haiku")
    
    # Pass the options object to query() using the options parameter
    async for message in query(
        prompt="What is the difference between Claude and Claude Code?",
        options=options
    ):
        # Display assistant message responses
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text)
        
        # Display usage metrics
        elif isinstance(message, ResultMessage):
            print(f"\n--- Usage Metrics ---")
            print(f"Cost: ${message.total_cost_usd:.4f}")
            print(f"Input tokens: {message.usage.get('input_tokens', 0)}")
            print(f"Output tokens: {message.usage.get('output_tokens', 0)}")


if __name__ == "__main__":
    anyio.run(main)

```

### Architectural Highlights:

* **The `options` Keyword Mapping:** Passing `options=options` overrides the runtime's default configurations for this single execution lifecycle block.
* **Cost Efficiency Optimization:** When checking the telemetry metrics inside the `ResultMessage` console block at runtime, you will see a dramatically lower cost-per-token footprint compared to running heavier models like Sonnet or Opus.

## Comparing Model Performance and Cost

Now that you understand how to configure agents with ClaudeAgentOptions, it's time to experience firsthand how model selection impacts both response quality and cost.

In this exercise, you'll compare the behavior and cost of two different Claude models: Haiku (the fastest and most cost-effective option) and Opus (the most capable model in the Claude family, with the highest reasoning ability).

Your task has two steps:

Step 1: Run the given code as-is with the Haiku model. Pay attention to the response content, structure, and especially the cost metrics displayed at the end. Write down or take note of the cost value.

Step 2: Change the model parameter in the ClaudeAgentOptions from "haiku" to "opus". Keep everything else the same and run the code again to observe the new response and cost.

Compare the two runs side-by-side. Notice the differences in response depth and explanation quality, but pay special attention to the cost difference — you'll see that Opus can cost significantly more than Haiku for the same task. This hands-on comparison will help you understand the critical trade-offs between model capability and cost, guiding your future model selection decisions based on both quality requirements and budget constraints!

```
import anyio
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ResultMessage,
)


async def main():
    # Configure the agent with ClaudeAgentOptions
    options = ClaudeAgentOptions(
        model="haiku"  # TODO: Change the model from haiku to opus
    )
    
    # Pass options to query() to apply configuration
    async for message in query(
        prompt="What is the difference between Claude and Claude Code?",
        options=options
    ):
        
        # Display assistant message responses
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text)
                    
        # Display usage metrics
        elif isinstance(message, ResultMessage):
            print(f"\n--- Usage Metrics ---")
            print(f"Cost: ${message.total_cost_usd:.4f}")
            print(f"Input tokens: {message.usage.get('input_tokens', 0)}")
            print(f"Output tokens: {message.usage.get('output_tokens', 0)}")


if __name__ == "__main__":
    anyio.run(main)

```

To complete **Step 2** of the exercise and switch your engine over to the premium reasoning tier, update the `model` string value inside your configuration block.

Here is the updated code configured to use the **Opus** model family:

```python
import anyio
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ResultMessage,
)


async def main():
    # Configure the agent with ClaudeAgentOptions
    options = ClaudeAgentOptions(
        model="opus"  # Changed the model from haiku to opus
    )
    
    # Pass options to query() to apply configuration
    async for message in query(
        prompt="What is the difference between Claude and Claude Code?",
        options=options
    ):
        
        # Display assistant message responses
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text)
                    
        # Display usage metrics
        elif isinstance(message, ResultMessage):
            print(f"\n--- Usage Metrics ---")
            print(f"Cost: ${message.total_cost_usd:.4f}")
            print(f"Input tokens: {message.usage.get('input_tokens', 0)}")
            print(f"Output tokens: {message.usage.get('output_tokens', 0)}")


if __name__ == "__main__":
    anyio.run(main)

```

### What you will observe from this comparison:

* **Response Depth:** The **Opus** run will typically output a significantly more detailed, deeply structured conceptual breakdown with sophisticated framing compared to the simpler, direct summary generated by **Haiku**.
* **Financial Metrics Shift:** When checking the `--- Usage Metrics ---` section at the end of the script execution, look closely at the `Cost` value. You will see that running the exact same prompt with **Opus** costs substantially more than it did with **Haiku**.

This trade-off is a core design decision when building agent platforms: you should save your heavy flagship models (Opus/Sonnet) for complex multi-step reasoning or codebase refactoring tasks, while routing basic lookups or classification turns through faster, cheaper models (Haiku) to optimize application spend!

## Configuring a Terse Engineer Persona

Now that you've seen how system prompts shape an agent's personality in the lesson, it's time to create a completely different character for your agent.

Your task is to add a system_prompt parameter to the ClaudeAgentOptions configuration that transforms the agent into a terse senior engineer who values brevity. This will create a stark contrast with the enthusiastic, beginner-friendly tutor from the lesson — instead of detailed explanations with emojis and tables, you'll get concise, technical responses that get straight to the point.

You should then observe how dramatically the same question produces a different response when the agent adopts this new personality!

```
import anyio
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
)


async def main():
    # TODO: Add a system_prompt parameter that configures the agent as a terse senior engineer who values brevity
    options = ClaudeAgentOptions(
        model="haiku"
    )
    
    # Pass options to query() to apply configuration
    async for message in query(
        prompt="What is the difference between Claude and Claude Code?",
        options=options
    ):
        
        # Display assistant message responses
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text)


if __name__ == "__main__":
    anyio.run(main)

```

Here is the updated code with the `system_prompt` parameter added to your `ClaudeAgentOptions` configuration.

This sets up a concise persona that tells the model to skip the introductory fluff and get straight to the technical differences:

```python
import anyio
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
)


async def main():
    # Add a system_prompt parameter that configures the agent as a terse senior engineer who values brevity
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are a terse senior engineer who values brevity. Get straight to the point without any pleasantries or fluff."
    )
    
    # Pass options to query() to apply configuration
    async for message in query(
        prompt="What is the difference between Claude and Claude Code?",
        options=options
    ):
        
        # Display assistant message responses
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text)


if __name__ == "__main__":
    anyio.run(main)

```

### What to Look For:

When you run this script, you will see a massive shift in how the response is framed. Instead of an encouraging, tutorial-style explanation with introductory greetings and multiple emojis, the output will drop down to a brief, highly dense technical comparison (often just a few bullet points), proving how strongly the system prompt directs the generation lifecycle!

## Observing Agent Behavior at Turn Limits

Now that you understand how to configure agents with ClaudeAgentOptions, let's explore what happens when an agent runs out of reasoning cycles before completing its task.

In this exercise, you'll intentionally create a situation in which the agent hits the max_turns limit. This is important to understand because, in real applications, you need to know how agents behave when they exhaust their allowed turns — whether they fail gracefully, return partial results, or clearly signal the limitation.

Your tasks are to:

    Add max_turns=2 to the ClaudeAgentOptions configuration
    Replace the prompt with one that asks the agent to write a short paragraph about Claude Code in a txt file

This multi-step prompt combined with a tight turn limit will force the agent to hit its maximum turns before finishing. The agent will attempt to use tools to write the file (even though we haven't configured tools yet), and these attempts will consume its limited turns. Run the code and observe the behavior — understanding how agents handle turn limits will help you set appropriate constraints in your own applications.

```
import anyio
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ResultMessage,
)


async def main():
    # Configure the agent with ClaudeAgentOptions
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are an enthusiastic beginner friendly tutor.",
    )
    
    # TODO: Change this prompt to ask the agent to write a short paragraph about Claude Code in a txt file
    async for message in query(
        prompt="Write a short paragraph about Claude Code,",
        options=options
    ):
        
        # Display assistant message responses
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text)
                    
        # Display result message
        elif isinstance(message, ResultMessage):
            print("\nResultMessage:")
            print(message)


if __name__ == "__main__":
    anyio.run(main)

```

Here is the completed code with `max_turns=2` injected into the configuration and the prompt updated to request a file creation action.

This multi-step task forces the agent to analyze the request and exhaust its turn budget attempting to plan or execute file actions before it can successfully finish the loop:

```python
import anyio
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ResultMessage,
)


async def main():
    # Configure the agent with ClaudeAgentOptions and a strict turn boundary
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are an enthusiastic beginner friendly tutor.",
        max_turns=2  # Added tight turn constraint
    )
    
    # Updated the prompt to request writing a paragraph about Claude Code into a .txt file
    async for message in query(
        prompt="Please write a short paragraph about Claude Code in a file named description.txt",
        options=options
    ):
        
        # Display assistant message responses
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text)
                    
        # Display result message
        elif isinstance(message, ResultMessage):
            print("\nResultMessage:")
            print(message)


if __name__ == "__main__":
    anyio.run(main)

```

### What to Look For in the Output:

When you run this script, the agent will likely hit its turn limit while processing the tool use lifecycle. Once it reaches turn 2, the execution halts gracefully. You will see a `ResultMessage` print at the bottom containing `subtype='error_max_turns'`, showing exactly how the SDK signals that the task was terminated due to resource constraints!